# EcoHome Energy Advisor - Agent Run & Evaluation

In this notebook, you'll run the Energy Advisor agent with various real-world scenarios and see how it helps customers optimize their energy usage.

## Learning Objectives
- Create the agent's instructions
- Run the Energy Advisor with different types of questions
- Evaluate response quality and accuracy
- Measure tool usage effectiveness
- Identify areas for improvement
- Implement evaluation metrics

## Evaluation Criteria
- **Accuracy**: Correct information and calculations
- **Relevance**: Responses address the user's question
- **Completeness**: Comprehensive answers with actionable advice
- **Tool Usage**: Appropriate use of available tools
- **Reasoning**: Clear explanation of recommendations


## 1. Import and Initialize

In [1]:
from datetime import datetime
from agent import Agent

In [2]:
## TODO: Create the agent's instructions
ECOHOME_SYSTEM_PROMPT = """You are the EcoHome Energy Advisor, helping homeowners optimize energy usage and reduce electricity costs.

## Your Tools
1. **get_weather_forecast** - Get weather and solar irradiance predictions
2. **get_electricity_prices** - Get hourly electricity rates (off-peak, peak, super-peak)
3. **query_energy_usage** - Get historical energy consumption data
4. **query_solar_generation** - Get solar panel production history
5. **search_energy_tips** - Find energy-saving tips from knowledge base
6. **calculate_energy_savings** - Calculate potential savings
7. **get_recent_energy_summary** - Get quick overview of recent energy usage (last 24h)

## When to Use Each Tool

**EV Charging Questions** → get_electricity_prices (+ get_weather_forecast if solar mentioned)
**Thermostat Questions** → get_weather_forecast (+ get_electricity_prices if cost mentioned)
**Appliance Scheduling** → get_electricity_prices
**Solar Predictions** → get_weather_forecast
**Solar + Grid Decisions** → get_electricity_prices + get_weather_forecast
**Historical Savings** → query_solar_generation + query_energy_usage
**General Tips/Best Practices** → search_energy_tips
**Quick Status/Overview** → get_recent_energy_summary
**"How much today/recently?"** → get_recent_energy_summary
**"Which device uses most?"** → get_recent_energy_summary
**"Tips for..." / "How to optimize..."** → search_energy_tips

## Electricity Rate Structure
- **Off-Peak (10 PM - 6 AM)**: ~$0.08/kWh - CHEAPEST
- **Peak (6 AM - 4 PM)**: ~$0.18/kWh
- **Super-Peak (4 PM - 9 PM)**: ~$0.30/kWh - MOST EXPENSIVE

## Response Guidelines
1. ALWAYS use appropriate tools before answering
2. Include SPECIFIC times (e.g., "between 10 PM and 6 AM")
3. Include COST estimates (e.g., "$0.08/kWh", "save $5/day")
4. For daytime activities, mention solar power benefits
5. Explain your reasoning based on the data

## Example Responses

**For EV charging:** "Charge your EV between 10 PM and 6 AM when rates are lowest at $0.08/kWh. For 50 kWh, this costs approximately $4.00 compared to $15.00 during super-peak hours."

**For thermostats:** "Based on tomorrow's forecast of 28°C, set your thermostat to 24°C. Pre-cool your home before 4 PM to avoid super-peak rates."

**For appliances:** "Run your dishwasher after 10 PM during off-peak hours to save approximately $0.22 per cycle."

**For energy overview:** "In the last 24 hours, you consumed 45.2 kWh total. Your HVAC used the most at 18.5 kWh ($4.25). Solar generated 12.3 kWh, offsetting $2.80 in costs."

**For energy tips:** "To reduce AC costs: 1) Set thermostat to 25°C instead of 22°C (saves ~15%), 2) Use ceiling fans to feel 3°C cooler, 3) Pre-cool before 4 PM super-peak hours."

Always be specific with times, temperatures, and dollar amounts.
"""

In [3]:
ecohome_agent = Agent(
    instructions=ECOHOME_SYSTEM_PROMPT,
)

In [4]:
response = ecohome_agent.invoke(
    question="When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
    context="Location: San Francisco, CA"
)

In [5]:
print(response["messages"][-1].content)

To minimize costs and maximize solar power when charging your electric car tomorrow in San Francisco, follow these recommendations:

### Best Charging Times:
1. **Off-Peak Hours (10 PM - 6 AM)**: 
   - **Cost**: $0.08/kWh
   - **Recommendation**: Charge your EV during these hours to take advantage of the lowest rates.

2. **Solar Power Utilization (10 AM - 3 PM)**:
   - **Best Solar Hours**: The solar generation is expected to be highest between **10 AM and 3 PM**. During this time, you can also benefit from solar energy if you have solar panels installed.
   - **Cost**: While the rates during this time are **$0.18/kWh**, if you can offset some of this with solar generation, it can be cost-effective.

### Summary:
- **Charge your EV overnight** from **10 PM to 6 AM** for the cheapest rate of **$0.08/kWh**.
- If you prefer to charge during the day, aim for **10 AM to 3 PM** to utilize solar power, but be aware of the higher rate of **$0.18/kWh**.

By charging overnight, you can save sig

In [6]:
print("TOOLS:")
for msg in response["messages"]:
    obj = msg.model_dump()
    if obj.get("tool_call_id"):
        print("-", msg.name)

TOOLS:
- get_weather_forecast
- get_electricity_prices


## 2. Define Test Cases

In [7]:
# TODO: Define comprehensive test cases for the Energy Advisor
# Create 10 test cases covering different scenarios:
# - EV charging optimization
# - Thermostat settings
# - Appliance scheduling
# - Solar power maximization
# - Cost savings calculations

In [8]:
test_cases = [
    {
        "id": "ev_charging_1",
        "question": "When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices"],
        "expected_response": "The response should contain time recommendation, cost analysis and solar consideration",
    },
    {
        "id": "ev_charging_2",
        "question": "My EV needs 50 kWh to fully charge. What's the cheapest time window tonight?",
        "expected_tools": ["get_electricity_prices"],
        "expected_response": "The response should include specific hours, estimated cost, and duration needed",
    },
    {
        "id": "thermostat_1",
        "question": "How should I set my thermostat for tomorrow given the weather forecast?",
        "expected_tools": ["get_weather_forecast"],
        "expected_response": "The response should include temperature settings and time-based adjustments",
    },
    {
        "id": "thermostat_2",
        "question": "Can I save money by pre-cooling my house before peak electricity hours?",
        "expected_tools": ["get_electricity_prices", "get_weather_forecast"],
        "expected_response": "The response should analyze pre-cooling strategy with cost comparison",
    },
    {
        "id": "appliance_1",
        "question": "When is the best time to run my dishwasher and laundry today?",
        "expected_tools": ["get_electricity_prices"],
        "expected_response": "The response should recommend specific hours aligned with low rates or solar availability",
    },
    {
        "id": "appliance_2",
        "question": "I need to run my pool pump for 4 hours. When should I schedule it?",
        "expected_tools": ["get_electricity_prices"],
        "expected_response": "The response should suggest optimal time window for extended appliance use",
    },
    {
        "id": "solar_1",
        "question": "How much solar power can I expect to generate this week?",
        "expected_tools": ["get_weather_forecast"],
        "expected_response": "The response should include daily estimates based on weather and solar irradiance",
    },
    {
        "id": "solar_2",
        "question": "Should I sell excess solar back to the grid or store it in my battery today?",
        "expected_tools": ["get_electricity_prices", "get_weather_forecast"],
        "expected_response": "The response should compare grid sell-back rates vs. future usage value",
    },
    {
        "id": "savings_1",
        "question": "How much money did I save last month by using solar power?",
        "expected_tools": ["query_solar_generation", "query_energy_usage"],
        "expected_response": "The response should show calculated savings with breakdown",
    },
    {
        "id": "savings_2",
        "question": "What are the top 3 ways I can reduce my electricity bill next month?",
        "expected_tools": ["get_electricity_prices"],
        "expected_response": "The response should provide prioritized actionable recommendations with estimated savings",
    },
    {
        "id": "summary_1",
        "question": "Give me a quick overview of my energy usage in the last 24 hours.",
        "expected_tools": ["get_recent_energy_summary"],
        "expected_response": "The response should include total consumption, device breakdown, and cost estimate",
    },
    {
        "id": "summary_2",
        "question": "Which device consumed the most electricity today and how much did it cost?",
        "expected_tools": ["get_recent_energy_summary"],
        "expected_response": "The response should identify the highest consuming device with kWh and cost",
    },

    # search_energy_tips Tests
    {
        "id": "tips_1",
        "question": "What are some tips for reducing my air conditioning costs?",
        "expected_tools": ["search_energy_tips"],
        "expected_response": "The response should provide practical HVAC efficiency tips and potential savings",
    },
    {
        "id": "tips_2",
        "question": "How can I optimize my electric vehicle charging to save money?",
        "expected_tools": ["search_energy_tips"],
        "expected_response": "The response should include EV charging best practices and cost-saving strategies",
    },
]


if len(test_cases) < 10:
    raise ValueError("You MUST have at least 10 test cases")

## 3. Run Agent Tests

In [9]:
CONTEXT = "Location: San Francisco, CA"

In [10]:
# Run the agent tests
# For each test case, call the agent and collect the response
# Store results for evaluation

print("=== Running Agent Tests ===")
test_results = []

for i, test_case in enumerate(test_cases):
    print(f"\nTest {i+1}: {test_case['id']}")
    print(f"Question: {test_case['question']}")
    print("-" * 50)
    
    try:
        # Call the agent
        response = ecohome_agent.invoke(
            question=test_case['question'],
            context=CONTEXT
        )
        
        # Store the result
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'response': response,
            'expected_tools': test_case['expected_tools'],
            'expected_response': test_case['expected_response'],
            'timestamp': datetime.now().isoformat()
        }
        test_results.append(result)
                
    except Exception as e:
        print(f"Error: {e}")
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'response': f"Error: {str(e)}",
            'expected_tools': test_case['expected_tools'],
            'expected_response': test_case['expected_response'],
            'timestamp': datetime.now().isoformat(),
            'error': str(e)
        }
        test_results.append(result)

print(f"\nCompleted {len(test_results)} tests")


=== Running Agent Tests ===

Test 1: ev_charging_1
Question: When should I charge my electric car tomorrow to minimize cost and maximize solar power?
--------------------------------------------------

Test 2: ev_charging_2
Question: My EV needs 50 kWh to fully charge. What's the cheapest time window tonight?
--------------------------------------------------

Test 3: thermostat_1
Question: How should I set my thermostat for tomorrow given the weather forecast?
--------------------------------------------------

Test 4: thermostat_2
Question: Can I save money by pre-cooling my house before peak electricity hours?
--------------------------------------------------

Test 5: appliance_1
Question: When is the best time to run my dishwasher and laundry today?
--------------------------------------------------

Test 6: appliance_2
Question: I need to run my pool pump for 4 hours. When should I schedule it?
--------------------------------------------------

Test 7: solar_1
Question: How much

In [11]:
test_results

[{'test_id': 'ev_charging_1',
  'question': 'When should I charge my electric car tomorrow to minimize cost and maximize solar power?',
  'response': {'messages': [SystemMessage(content='Location: San Francisco, CA', additional_kwargs={}, response_metadata={}, id='4049c75d-a056-4ba8-aa23-9e364a12d051'),
    HumanMessage(content='When should I charge my electric car tomorrow to minimize cost and maximize solar power?', additional_kwargs={}, response_metadata={}, id='464a85e4-deee-4c97-bbc7-cc9824bf4149'),
    AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 1433, 'total_tokens': 1494, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_373a14eb6f', 'id': 'ch

## 4. Evaluate Responses

In [12]:
# TODO: Implement evaluation functions
# Create functions to evaluate:
# - Final Response
# - Tool usage

In [13]:
# TODO: Create a response evaluator
def evaluate_response(question, final_response, expected_response):
    """Evaluate if the response meets the expected criteria."""

    if not final_response:
        return {
            "score": 0.0,
            "passed": False,
            "checks": {},
            "feedback": ["No response provided"]
        }

    question_lower = question.lower()
    expected_lower = expected_response.lower()
    response_lower = final_response.lower()

    checks = {}
    feedback = []

    # Check question relevance using key terms from the question
    question_terms = {
        "charge": ["charge", "charging", "ev", "electric car", "vehicle"],
        "thermostat": ["thermostat", "temperature", "hvac", "heating", "cooling", "degrees"],
        "solar": ["solar", "sun", "generation", "panel", "irradiance"],
        "appliance": ["appliance", "dishwasher", "laundry", "washing", "dryer", "pool pump"],
        "save": ["save", "saving", "cost", "bill", "reduce", "money"],
        "weather": ["weather", "forecast", "sunny", "cloudy", "rain"]
    }

    # Find which category the question belongs to
    relevance_score = 0
    for category, terms in question_terms.items():
        if any(term in question_lower for term in terms):
            # Check if response addresses this category
            if any(term in response_lower for term in terms):
                relevance_score += 1

    checks["question_relevance"] = relevance_score > 0

    # Define keyword mappings for expected criteria
    criteria_keywords = {
        "time recommendation": ["hour", "time", "am", "pm", "morning", "afternoon", "evening", "night",
                                "00:", "01:", "02:", "03:", "04:", "05:", "06:", "07:", "08:", "09:",
                                "10:", "11:", "12:", "13:", "14:", "15:", "16:", "17:", "18:", "19:",
                                "20:", "21:", "22:", "23:", "o'clock", "midnight", "noon",
                                "off-peak", "peak", "schedule"],
        "cost analysis": ["$", "cost", "rate", "price", "kwh", "per", "save", "saving", "cheap",
                          "expensive", "bill", "dollar", "cent", "usd"],
        "solar consideration": ["solar", "sun", "panel", "generation", "irradiance", "renewable",
                                "photovoltaic", "pv", "daylight"],
        "weather": ["weather", "temperature", "sunny", "cloudy", "forecast", "humidity", "degrees",
                    "celsius", "fahrenheit", "warm", "cold", "hot"],
        "savings": ["save", "saving", "reduce", "cut", "lower", "minimize", "optimize", "efficiency",
                    "annual", "monthly", "daily", "$"],
        "specific hours": ["10 pm", "10pm", "22:00", "6 am", "6am", "06:00", "off-peak",
                           "midnight", "night", "morning"],
        "estimated cost": ["$", "dollar", "usd", "cost", "estimate", "approximately", "about", "around"]
    }

    # Check which criteria are expected based on expected_response
    matched_criteria = 0
    total_criteria = 0

    for criterion, keywords in criteria_keywords.items():
        # Check if this criterion is expected
        criterion_words = criterion.replace("_", " ").split()
        if any(word in expected_lower for word in criterion_words) or \
                any(kw in expected_lower for kw in keywords[:3]):
            total_criteria += 1
            found = any(kw in response_lower for kw in keywords)
            checks[criterion] = found
            if found:
                matched_criteria += 1
            else:
                feedback.append(f"Missing: {criterion}")

    # Calculate score
    if total_criteria > 0:
        criteria_score = matched_criteria / total_criteria
    else:
        criteria_score = 1.0 if len(final_response) > 100 else 0.5

    # Combine relevance and criteria scores
    relevance_weight = 0.3
    criteria_weight = 0.7

    relevance_value = 1.0 if checks.get("question_relevance", False) else 0.0
    score = (relevance_value * relevance_weight) + (criteria_score * criteria_weight)

    passed = score >= 0.6

    if passed:
        feedback.append("Response meets expected criteria")
    if not checks.get("question_relevance", True):
        feedback.append("Response may not be relevant to the question")

    return {
        "score": round(score, 2),
        "passed": passed,
        "checks": checks,
        "feedback": feedback
    }

### Evaluate first test

In [14]:
# Evaluate first test
test_result = test_results[0]
test_case = test_cases[0]

# Extract final response from message chain
messages = test_result["response"]["messages"]

# Find last AIMessage with content
final_response = ""
for msg in reversed(messages):
    if hasattr(msg, "content") and msg.content and hasattr(msg, "tool_calls"):
        # AIMessage with content and no active tool calls
        if not msg.tool_calls:
            final_response = msg.content
            break

# Run evaluation
evaluation = evaluate_response(
    question=test_case["question"],
    final_response=final_response,
    expected_response=test_case["expected_response"]
)

print(f"Test: {test_case['id']}")
print(f"Score: {evaluation['score']}")
print(f"Passed: {'✅' if evaluation['passed'] else '❌'}")
print(f"Checks: {evaluation['checks']}")
print(f"Feedback: {evaluation['feedback']}")
print(f"\nResponse (excerpt): {final_response[:300]}...")

Test: ev_charging_1
Score: 1.0
Passed: ✅
Checks: {'question_relevance': True, 'time recommendation': True, 'cost analysis': True, 'solar consideration': True, 'estimated cost': True}
Feedback: ['Response meets expected criteria']

Response (excerpt): To minimize costs and maximize solar power when charging your electric vehicle (EV) tomorrow in San Francisco, follow these recommendations:

### Best Charging Times:
1. **Off-Peak Hours (10 PM - 6 AM)**: 
   - **Charge your EV between 10 PM and 6 AM** when electricity rates are the lowest at approx...


### Evaluate all tests

In [15]:
all_evaluations = []

for i, test_result in enumerate(test_results):
    test_case = test_cases[i]
    messages = test_result["response"]["messages"]

    # Extract final response
    final_response = ""
    for msg in reversed(messages):
        if hasattr(msg, "content") and msg.content:
            if hasattr(msg, "tool_calls") and not msg.tool_calls:
                final_response = msg.content
                break

    evaluation = evaluate_response(
        question=test_case["question"],
        final_response=final_response,
        expected_response=test_case["expected_response"]
    )

    all_evaluations.append(evaluation)
    status = "✅" if evaluation["passed"] else "❌"
    print(f"{status} {test_case['id']}: Score {evaluation['score']}")

# Summary statistics
total = len(all_evaluations)
passed = sum(1 for e in all_evaluations if e["passed"])
avg_score = sum(e["score"] for e in all_evaluations) / total if total > 0 else 0

print(f"\n{'='*50}")
print(f"SUMMARY")
print(f"Total Tests: {total}")
print(f"Passed: {passed}/{total} ({100*passed/total:.1f}%)")
print(f"Average Score: {avg_score:.2f}")
print(f"{'='*50}")

✅ ev_charging_1: Score 1.0
✅ ev_charging_2: Score 1.0
✅ thermostat_1: Score 1.0
✅ thermostat_2: Score 1.0
✅ appliance_1: Score 1.0
✅ appliance_2: Score 1.0
✅ solar_1: Score 0.65
✅ solar_2: Score 1.0
✅ savings_1: Score 1.0
✅ savings_2: Score 1.0
✅ summary_1: Score 0.7
✅ summary_2: Score 1.0
✅ tips_1: Score 1.0
✅ tips_2: Score 1.0

SUMMARY
Total Tests: 14
Passed: 14/14 (100.0%)
Average Score: 0.95


In [16]:
# TODO: Create a tool udage evaluator
def evaluate_tool_usage(messages, expected_tools):
    """Evaluate if expected tools were called correctly."""
    tools_called = []

    for msg in messages:
        # Extract tool names from AIMessage tool_calls
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tool_call in msg.tool_calls:
                if isinstance(tool_call, dict):
                    tool_name = tool_call.get("name")
                elif hasattr(tool_call, "name"):
                    tool_name = tool_call.name
                else:
                    continue
                if tool_name and tool_name not in tools_called:
                    tools_called.append(tool_name)

    # Compare with expected tools
    expected_set = set(expected_tools)
    called_set = set(tools_called)

    missing = expected_set - called_set
    extra = called_set - expected_set
    matched = expected_set & called_set

    # Calculate score
    if len(expected_tools) > 0:
        score = len(matched) / len(expected_tools)
    else:
        score = 1.0 if len(tools_called) == 0 else 0.5

    passed = len(missing) == 0

    feedback = []
    if missing:
        feedback.append(f"Missing expected tools: {list(missing)}")
    if extra:
        feedback.append(f"Additional tools used: {list(extra)}")
    if passed:
        feedback.append("All expected tools were called")

    return {
        "score": score,
        "passed": passed,
        "tools_called": tools_called,
        "feedback": feedback
    }


### Evaluate tool usage for first test

In [17]:
# Evaluate tool usage for first test
test_result = test_results[0]
test_case = test_cases[0]

# Extract messages from response
messages = test_result["response"]["messages"]

# Run tool usage evaluation
tool_evaluation = evaluate_tool_usage(
    messages=messages,
    expected_tools=test_case["expected_tools"]
)

print(f"Test: {test_case['id']}")
print(f"Score: {tool_evaluation['score']}")
print(f"Passed: {'✅' if tool_evaluation['passed'] else '❌'}")
print(f"Expected Tools: {test_case['expected_tools']}")
print(f"Tools Called: {tool_evaluation.get('tools_called', [])}")
print(f"Feedback: {tool_evaluation['feedback']}")

Test: ev_charging_1
Score: 1.0
Passed: ✅
Expected Tools: ['get_weather_forecast', 'get_electricity_prices']
Tools Called: ['get_weather_forecast', 'get_electricity_prices']
Feedback: ['All expected tools were called']


In [18]:
# TODO: Generate a comprehensive evaluation report
# Calculate overall scores and metrics
# Identify strengths and weaknesses
# Provide recommendations for improvement
def print_evaluation_report(report):
    """Generate a comprehensive evaluation report."""

    print("=" * 60)
    print("COMPREHENSIVE EVALUATION REPORT")
    print("=" * 60)

    # Overall metrics
    total_tests = len(report)

    # Response evaluation metrics
    response_scores = [r.get("response_evaluation", {}).get("score", 0) for r in report]
    response_passed = sum(1 for r in report if r.get("response_evaluation", {}).get("passed", False))
    avg_response_score = sum(response_scores) / total_tests if total_tests > 0 else 0

    # Tool usage metrics
    tool_scores = [r.get("tool_evaluation", {}).get("score", 0) for r in report]
    tool_passed = sum(1 for r in report if r.get("tool_evaluation", {}).get("passed", False))
    avg_tool_score = sum(tool_scores) / total_tests if total_tests > 0 else 0

    # Overall score (combined)
    overall_score = (avg_response_score + avg_tool_score) / 2

    print(f"OVERALL METRICS")
    print(f"   Total Tests: {total_tests}")
    print(f"   Overall Score: {overall_score:.2f}")
    print(f"   Response Score: {avg_response_score:.2f} ({response_passed}/{total_tests} passed)")
    print(f"   Tool Usage Score: {avg_tool_score:.2f} ({tool_passed}/{total_tests} passed)")

    # Identify strengths
    print(f"STRENGTHS")
    strengths = []
    if avg_tool_score >= 0.8:
        strengths.append("Excellent tool selection and usage")
    if avg_response_score >= 0.8:
        strengths.append("High-quality response generation")
    if response_passed == total_tests:
        strengths.append("All responses meet expected criteria")
    if not strengths:
        strengths.append("No significant strengths identified")
    for s in strengths:
        print(f"   • {s}")

    # Identify weaknesses
    print(f"WEAKNESSES")
    weaknesses = []
    if avg_tool_score < 0.7:
        weaknesses.append("Tool selection needs improvement")
    if avg_response_score < 0.7:
        weaknesses.append("Response quality below threshold")

    # Analyze specific failures
    failed_checks = {}
    for r in report:
        checks = r.get("response_evaluation", {}).get("checks", {})
        for check, passed in checks.items():
            if not passed:
                failed_checks[check] = failed_checks.get(check, 0) + 1

    for check, count in failed_checks.items():
        weaknesses.append(f"'{check}' failed in {count} test(s)")

    if not weaknesses:
        weaknesses.append("No significant weaknesses identified")
    for w in weaknesses:
        print(f"   • {w}")

    # Recommendations
    print(f"RECOMMENDATIONS")
    recommendations = []
    if avg_tool_score < 0.8:
        recommendations.append("Review tool definitions and ensure agent understands when to use each tool")
    if avg_response_score < 0.8:
        recommendations.append("Improve system prompt to generate more comprehensive responses")
    if "question_relevance" in failed_checks:
        recommendations.append("Ensure responses directly address the user's question")
    if overall_score >= 0.9:
        recommendations.append("Agent performs well - consider adding more complex test cases")
    if not recommendations:
        recommendations.append("Continue monitoring performance with additional test cases")
    for rec in recommendations:
        print(f"   • {rec}")

    # Detailed test results
    print(f"DETAILED RESULTS")
    print("-" * 60)
    for r in report:
        test_id = r.get("test_id", "unknown")
        resp_eval = r.get("response_evaluation", {})
        tool_eval = r.get("tool_evaluation", {})

        resp_status = "✅" if resp_eval.get("passed") else "❌"
        tool_status = "✅" if tool_eval.get("passed") else "❌"

        print(f"   {test_id}:")
        print(f"      Response: {resp_status} (Score: {resp_eval.get('score', 0):.2f})")
        print(f"      Tools: {tool_status} (Score: {tool_eval.get('score', 0):.2f})")

    print("=" * 60)


### Report

In [19]:
# Build the report from test results
report = []

for i, test_result in enumerate(test_results):
    test_case = test_cases[i]
    messages = test_result["response"]["messages"]

    # Get final response (last AI message content)
    final_response = ""
    for msg in reversed(messages):
        if hasattr(msg, "content") and msg.content and not hasattr(msg, "tool_calls"):
            final_response = msg.content
            break

    # Evaluate tool usage
    tool_evaluation = evaluate_tool_usage(
        messages=messages,
        expected_tools=test_case["expected_tools"]
    )

    # Evaluate response
    response_evaluation = evaluate_response(
        question=test_case["question"],
        final_response=final_response,
        expected_response=test_case["expected_response"]
    )

    report.append({
        "test_id": test_case["id"],
        "question": test_case["question"],
        "final_response": final_response,
        "response_evaluation": response_evaluation,
        "tool_evaluation": tool_evaluation
    })

# Print the comprehensive report
print_evaluation_report(report)

COMPREHENSIVE EVALUATION REPORT
OVERALL METRICS
   Total Tests: 14
   Overall Score: 0.81
   Response Score: 0.76 (10/14 passed)
   Tool Usage Score: 0.86 (12/14 passed)
STRENGTHS
   • Excellent tool selection and usage
WEAKNESSES
   • 'solar consideration' failed in 1 test(s)
   • 'question_relevance' failed in 6 test(s)
   • 'specific hours' failed in 2 test(s)
   • 'estimated cost' failed in 3 test(s)
   • 'weather' failed in 1 test(s)
RECOMMENDATIONS
   • Improve system prompt to generate more comprehensive responses
   • Ensure responses directly address the user's question
DETAILED RESULTS
------------------------------------------------------------
   ev_charging_1:
      Response: ✅ (Score: 0.82)
      Tools: ✅ (Score: 1.00)
   ev_charging_2:
      Response: ❌ (Score: 0.35)
      Tools: ✅ (Score: 1.00)
   thermostat_1:
      Response: ❌ (Score: 0.35)
      Tools: ✅ (Score: 1.00)
   thermostat_2:
      Response: ✅ (Score: 1.00)
      Tools: ✅ (Score: 1.00)
   appliance_1:
      